# ML-003 — Baseline de classificação

Dois modelos, mesmo split treino/teste, para comparação direta:

1. **TF-IDF + LogisticRegression** (`class_weight="balanced"`) sobre `clinical_notes` — o baseline "de verdade".
2. **Baseline ingênuo** por `chief_complaint` — a feature que o [ADR-002](../docs/decisions/ADR-002-text-leakage.md) identificou como vazamento determinístico do rótulo.

Métricas: F1 macro + recall por classe (com foco em `urgente`) — acurácia sozinha não é confiável aqui (ver ADR-002: o dataset infla acurácia por construção).

Card: [docs/KANBAN.md](../docs/KANBAN.md) — ML-003.

## 1. Setup e dados

Carrega `data/raw/fedmml_ed_triage_raw.parquet` (gerado na ML-002) em vez do parquet processado — precisamos de `chief_complaint` além de `clinical_notes` para o baseline ingênuo, e os dois modelos devem ver exatamente o mesmo split para a comparação ser justa.

In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

RAW_PATH = "../data/raw/fedmml_ed_triage_raw.parquet"
MODELS_DIR = Path("../models")
EXPERIMENTS_DIR = Path("../docs/experiments")
CLASSES = ["normal", "atencao", "urgente"]
SEED = 42

In [2]:
def remap_esi(esi: int) -> str:
    if esi in (1, 2):
        return "urgente"
    if esi == 3:
        return "atencao"
    if esi in (4, 5):
        return "normal"
    raise ValueError(f"ESI fora do intervalo esperado (1-5): {esi!r}")


df = pd.read_parquet(RAW_PATH)
df["urgencia"] = df["esi_level"].map(remap_esi)

# ML-002: 1,78% de clinical_notes nulos — não dá para treinar/avaliar um
# classificador de texto nessas linhas.
df = df.dropna(subset=["clinical_notes"]).reset_index(drop=True)
df[["clinical_notes", "chief_complaint", "urgencia"]].shape

(85679, 3)

## 2. Split treino/teste (compartilhado pelos dois baselines)

In [3]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["urgencia"],
    random_state=SEED,
)
y_train, y_test = train_df["urgencia"], test_df["urgencia"]
len(train_df), len(test_df)

(68543, 17136)

## 3. Baseline 1 — TF-IDF + LogisticRegression

In [4]:
tfidf_pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
    ]
)
tfidf_pipeline.fit(train_df["clinical_notes"], y_train)
tfidf_preds = tfidf_pipeline.predict(test_df["clinical_notes"])

print(classification_report(y_test, tfidf_preds, labels=CLASSES))

              precision    recall  f1-score   support

      normal       1.00      1.00      1.00      5522
     atencao       1.00      1.00      1.00      8128
     urgente       1.00      1.00      1.00      3486

    accuracy                           1.00     17136
   macro avg       1.00      1.00      1.00     17136
weighted avg       1.00      1.00      1.00     17136



## 4. Baseline 2 — ingênuo por `chief_complaint`

"Treina" um lookup `chief_complaint → classe majoritária` só a partir do treino (metodologia correta, mesmo sabendo pelo ADR-002 que o mapeamento é determinístico). Fallback para a classe majoritária global cobre categorias eventualmente ausentes no treino.

In [5]:
complaint_to_class = (
    train_df.groupby("chief_complaint")["urgencia"]
    .agg(lambda s: s.value_counts().idxmax())
    .to_dict()
)
fallback_class = y_train.value_counts().idxmax()

naive_preds = test_df["chief_complaint"].map(complaint_to_class).fillna(fallback_class)

print(classification_report(y_test, naive_preds, labels=CLASSES))

              precision    recall  f1-score   support

      normal       1.00      1.00      1.00      5522
     atencao       1.00      1.00      1.00      8128
     urgente       1.00      1.00      1.00      3486

    accuracy                           1.00     17136
   macro avg       1.00      1.00      1.00     17136
weighted avg       1.00      1.00      1.00     17136



## 5. Comparação lado a lado

In [6]:
def build_metrics(y_true, y_pred, model_name: str) -> dict:
    recall_per_class = dict(
        zip(CLASSES, recall_score(y_true, y_pred, labels=CLASSES, average=None), strict=True)
    )
    return {
        "model": model_name,
        "n_test": int(len(y_true)),
        "f1_macro": float(f1_score(y_true, y_pred, labels=CLASSES, average="macro")),
        "recall_macro": float(recall_score(y_true, y_pred, labels=CLASSES, average="macro")),
        "recall_per_class": {k: float(v) for k, v in recall_per_class.items()},
        "recall_urgente": float(recall_per_class["urgente"]),
        "confusion_matrix": {
            "labels": CLASSES,
            "matrix": confusion_matrix(y_true, y_pred, labels=CLASSES).tolist(),
        },
    }


metrics = {
    "task": "ML-003",
    "split": {"test_size": 0.2, "stratify": "urgencia", "random_state": SEED},
    "models": {
        "tfidf_logreg": build_metrics(y_test, tfidf_preds, "TF-IDF + LogisticRegression"),
        "naive_chief_complaint": build_metrics(y_test, naive_preds, "Ingênuo (chief_complaint)"),
    },
}

pd.DataFrame(
    {
        name: {
            "f1_macro": m["f1_macro"],
            "recall_macro": m["recall_macro"],
            "recall_urgente": m["recall_urgente"],
        }
        for name, m in metrics["models"].items()
    }
).T.round(4)

,f1_macro,recall_macro,recall_urgente
tfidf_logreg,1.0,1.0,1.0
naive_chief_complaint,1.0,1.0,1.0


**Leitura**: os dois baselines batem exatamente 1.00 em F1 macro, recall macro e recall de `urgente` (17.136 exemplos de teste). Confirma a expectativa do ADR-002 na íntegra — o TF-IDF + LogisticRegression não supera o baseline ingênuo por `chief_complaint` porque não há nada a superar: o texto codifica a mesma informação determinística que a queixa, sem ruído adicional. Isso não é um bug do pipeline nem sorte de split; é a consequência direta e esperada da geração sintética do dataset (ver ADR-002 e a validação cruzada com vitais/labs na ML-002, que mostrou que esse determinismo é específico do texto).

## 6. Salvar artefatos

Modelos em `models/` (gitignored, regenerável rodando este notebook). Métricas em `docs/experiments/` (versionado — números prontos para a subseção "6.1 Limitações do dataset" do README, tarefa separada de documentação).

In [7]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(tfidf_pipeline, MODELS_DIR / "tfidf_logreg_baseline.joblib")
(MODELS_DIR / "naive_chief_complaint_baseline.json").write_text(
    json.dumps(
        {"complaint_to_class": complaint_to_class, "fallback_class": fallback_class}, indent=2
    )
)
(EXPERIMENTS_DIR / "ML-003-baseline-metrics.json").write_text(json.dumps(metrics, indent=2))

print("salvo:")
print(f"- {MODELS_DIR / 'tfidf_logreg_baseline.joblib'}")
print(f"- {MODELS_DIR / 'naive_chief_complaint_baseline.json'}")
print(f"- {EXPERIMENTS_DIR / 'ML-003-baseline-metrics.json'}")

salvo:
- ../models/tfidf_logreg_baseline.joblib
- ../models/naive_chief_complaint_baseline.json
- ../docs/experiments/ML-003-baseline-metrics.json
